# Project: Recruiter Pipeline Assistant
**Brief from Paras:**
Watches Gmail for resumes, parses, scores against JD in Notion,
schedules screening calls on Calendar for qualified candidates, updates Notion DB.

This is a SCAFFOLD. The supervisor + graph wiring is complete; worker logic for some nodes is marked TODO.
Use Project #2 (`02_support_resolver/support_resolver.ipynb`) as your reference implementation.

## Submission checklist
- [ ] All worker TODOs filled in
- [ ] Composio actions verified against docs.composio.dev
- [ ] HITL where destructive actions occur
- [ ] Checkpoint persistence configured
- [ ] Graph diagram saved as graph.png
- [ ] README.md with architecture + example run


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect required apps in their dashboard"
print("env OK")


## 2. State schema

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class RecruiterPipelineState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    resume_text: str
    candidate_email: str
    parsed: Optional[dict]
    score: Optional[float]
    calendar_event: Optional[str]


## 3. LLM and Composio toolset

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()


## 4. Workers

In [ ]:
# parser - Resume parser (pure LLM, no Composio)
# No Composio actions; pure LLM or custom logic.
# TODO: implement parser_node(state) following the pattern in 02_support_resolver.
def parser_node(state):
    """Extract skills list, years of experience, education from the resume text."""
    raise NotImplementedError('TODO: implement parser_node')


# scorer - JD scorer via Notion
scorer_tools = toolset.get_tools(actions=[Action.NOTION_QUERY_DATABASE])
scorer_agent = create_react_agent(llm, scorer_tools, prompt='''Pull the active JD from Notion. Score candidate 0-100 against requirements.''')
def scorer_node(state):
    result = scorer_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='scorer')]}


# scheduler - Calendar booker
scheduler_tools = toolset.get_tools(actions=[Action.GOOGLECALENDAR_CREATE_EVENT])
scheduler_agent = create_react_agent(llm, scheduler_tools, prompt='''If score >= 70, book a 30-min screening call next week.''')
def scheduler_node(state):
    result = scheduler_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='scheduler')]}


# db_updater - Notion candidate DB updater
db_updater_tools = toolset.get_tools(actions=[Action.NOTION_CREATE_NOTION_PAGE, Action.NOTION_APPEND_TEXT_BLOCKS])
db_updater_agent = create_react_agent(llm, db_updater_tools, prompt='''Write candidate row into 'Pipeline' Notion DB with score, link, status.''')
def db_updater_node(state):
    result = db_updater_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='db_updater')]}


## 5. Supervisor + router

In [ ]:
def supervisor(state) -> dict:
    if state.get('parsed') is None: return {'next_worker': 'parser'}
    if state.get('score') is None: return {'next_worker': 'scorer'}
    if state['score'] >= 70 and state.get('calendar_event') is None: return {'next_worker': 'scheduler'}
    if 'pipeline updated' not in (state['messages'][-1].content.lower() if state['messages'] else ''):
        return {'next_worker': 'db_updater'}
    return {'next_worker': 'DONE'}

def route(state) -> str:
    nxt = state['next_worker']
    if nxt in {'scorer', 'parser', 'db_updater', 'scheduler'}:
        return nxt
    return '__end__'


## 6. Build graph + checkpoint persistence

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(globals()[[k for k in globals() if k.endswith('State') and k != 'AgentState'][0]])
g.add_node("supervisor", supervisor)
g.add_node("parser", parser_node)
g.add_node("scorer", scorer_node)
g.add_node("scheduler", scheduler_node)
g.add_node("db_updater", db_updater_node)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "parser": "parser",
    "scorer": "scorer",
    "scheduler": "scheduler",
    "db_updater": "db_updater",
    "__end__": END,
})
g.add_edge("parser", "supervisor")
g.add_edge("scorer", "supervisor")
g.add_edge("scheduler", "supervisor")
g.add_edge("db_updater", "supervisor")

conn = sqlite3.connect("05_recruiter_pipeline.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 7. Visualise (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 8. End-to-end demo

In [ ]:
config = {'configurable': {'thread_id': '05_recruiter_pipeline-demo-1'}, 'recursion_limit': 30}

result = app.invoke(
    {'resume_text': 'Senior Python dev, 6 years experience...',
        'candidate_email': 'jane@example.com',
        'messages': [HumanMessage(content='New resume from Jane Doe')]},
    config=config,
)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != 'messages':
        print(f"{k}: {str(v)[:150]}")
print("\n=== MESSAGE TRACE ===")
for m in result['messages']:
    label = getattr(m, 'name', m.type)
    print(f"[{label}] {m.content[:200]}")
